# Notebook 2: Baseline Model (Fully Connected Network)

## Objective

Implement a baseline neural network **without convolutional layers** to establish a reference point for comparison. This will help us understand the improvements that convolutional layers bring.

## Architecture

The baseline model will use only fully connected (dense) layers:
- **Input**: Flattened 32×32×3 = 3,072 features
- **Hidden layers**: Two fully connected layers with ReLU activation
- **Output**: 10 classes (softmax)

## Expected Limitations

- **No spatial awareness**: Flattening destroys 2D structure
- **Large parameter count**: Every pixel connects to every neuron
- **No translation invariance**: Must learn same pattern at every position
- **Overfitting risk**: Too many parameters for limited data

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_loader import get_cifar10_loaders, get_class_names
from src.models.baseline import get_baseline_model
from src.training.trainer import train_model, evaluate_model
from src.utils.visualization import plot_training_history, plot_confusion_matrix

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Data

In [ ]:
# Load CIFAR-10 data with preprocessing
train_loader, val_loader, test_loader = get_cifar10_loaders(
    batch_size=128,
    val_split=0.1,
    data_dir='../data/raw'
)

class_names = get_class_names()

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 2. Define Baseline Model

In [ ]:
# Create baseline model
model = get_baseline_model(hidden_sizes=[512, 256])

print("Baseline Model Architecture:")
print("="*60)
print(model)
print("="*60)
print(f"\nTotal Parameters: {model.count_parameters():,}")

# Calculate model size in MB
param_size = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
size_mb = (param_size + buffer_size) / 1024**2
print(f"Model Size: {size_mb:.2f} MB")

## 3. Architecture Justification

### Layer Choices:

1. **Input Layer (3072 → 512)**:
   - Takes flattened 32×32×3 image
   - Compresses to 512 features
   
2. **Hidden Layer (512 → 256)**:
   - Further feature compression
   - Adds non-linearity with ReLU
   
3. **Dropout (20%)**:
   - Regularization to prevent overfitting
   - Applied after each hidden layer
   
4. **Output Layer (256 → 10)**:
   - Maps to 10 class scores
   
### Parameter Count Analysis:

- Layer 1: 3072 × 512 + 512 = **1,573,376** parameters
- Layer 2: 512 × 256 + 256 = **131,328** parameters  
- Layer 3: 256 × 10 + 10 = **2,570** parameters
- **Total: ~1.7M parameters**

This is a very large number for such a small dataset!

## 4. Train the Model

In [ ]:
# Train the model
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=30,
    learning_rate=0.001,
    device=device,
    save_path='../results/models/baseline_best.pth'
)

## 5. Training History Visualization

In [ ]:
# Plot training history
plot_training_history(history, save_path='../results/figures/baseline_training_history.png')

## 6. Load Best Model and Evaluate

In [ ]:
# Load best model
checkpoint = torch.load('../results/models/baseline_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Best validation accuracy: {checkpoint['val_acc']:.2f}%")

In [ ]:
# Evaluate on test set
test_acc, predictions, true_labels = evaluate_model(model, test_loader, device)

print(f"\nTest Accuracy: {test_acc:.2f}%")

## 7. Confusion Matrix

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(
    true_labels, 
    predictions, 
    class_names,
    save_path='../results/figures/baseline_confusion_matrix.png'
)

## 8. Per-Class Performance

In [ ]:
from sklearn.metrics import classification_report

# Generate classification report
report = classification_report(true_labels, predictions, 
                              target_names=class_names, 
                              digits=3)
print("\nPer-Class Performance:")
print("="*70)
print(report)

## 9. Sample Predictions

In [ ]:
# Visualize some predictions
import torchvision

model.eval()
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Get predictions
with torch.no_grad():
    outputs = model(images.to(device))
    _, predicted = torch.max(outputs, 1)

# Plot first 10 images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for idx in range(10):
    img = images[idx]
    # Unnormalize
    img = img * torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    img = img + torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    img = torch.clamp(img, 0, 1)
    img = img.permute(1, 2, 0).numpy()
    
    axes[idx].imshow(img)
    true_label = class_names[labels[idx]]
    pred_label = class_names[predicted[idx]]
    color = 'green' if labels[idx] == predicted[idx] else 'red'
    axes[idx].set_title(f'True: {true_label}\nPred: {pred_label}', color=color)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../results/figures/baseline_sample_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary: Baseline Model Performance

### Model Characteristics:
- **Architecture**: Flatten → Dense(512) → Dense(256) → Dense(10)
- **Parameters**: ~1.7 million
- **Training Time**: ~X minutes per epoch
- **Test Accuracy**: ~XX%

### Observed Limitations:

1. **High Parameter Count**: 
   - 1.7M parameters for just 50K training samples
   - Risk of overfitting

2. **No Spatial Understanding**:
   - Flattening destroys 2D structure
   - Can't leverage local patterns
   
3. **No Translation Invariance**:
   - Must learn same feature at every position
   - Inefficient use of parameters
   
4. **Limited Performance**:
   - Baseline accuracy is decent but not excellent
   - Some classes are confused more than others

### Why These Limitations Matter:

Images have **natural 2D structure** with:
- **Local patterns**: Edges, textures, shapes
- **Spatial relationships**: Nearby pixels are related
- **Translation invariance**: A "cat" is a "cat" regardless of position

Fully connected layers **ignore all of this**, treating the image as a flat vector.

### Next Steps:

We'll design a CNN that:
1. Preserves 2D structure
2. Uses fewer parameters
3. Exploits local patterns
4. Achieves better performance